# Phase C — Explanation Faithfulness (attribution protocol, agreement, perturbation)
### *Trustworthiness Under Compression* — does compression degrade the EXPLANATIONS too?

Implements the §4.0 locked attribution protocol and both faithfulness families on the seed-0 anchor's compression matrix (M1–M3), with the three-floor noise hierarchy.

**Pipeline**
1. **§4.0 protocol** — one model-agnostic explainer (KernelSHAP) applied identically to every model via a unified `predict_proba`; fixed stratified eval subset + k-means background, both seeded; attribute the **true class**; fixed `nsamples`.
2. **SHAP once, saved** — compute + cache attributions for: M0(seed0) ×2 (explainer-noise floor), M0 seed1/seed2 (Control-1 null band), M1_int8, M1_fp16, M2_50, M2_80, M3.
3. **§4.1 agreement** — Spearman, Kendall, Jaccard@k, sign-flip vs M0; per-class; judged against the null band + noise floor.
4. **§4.2 perturbation** — comprehensiveness / AOPC with averaged random-k baseline, under **three maskers** (mean/zero/marginal) for the masking-sensitivity check.

> SHAP is the slow step (once, cached). Knobs `N_PER_CLASS`, `NSAMPLES`, `BG_K` at top — defaults are tractable; raise for final numbers. GPU helps keras; TFLite is CPU.

## 0. Setup + load all models

In [1]:
!pip install -q shap
import os, gc, json, time, random
import numpy as np, pandas as pd, tensorflow as tf, joblib, shap
from scipy.stats import spearmanr, kendalltau

def seed_everything(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
seed_everything(0)

ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
if not os.path.isdir(f'{ROOT}/data'):
    from google.colab import drive; drive.mount('/content/drive')
PATHS={k:f'{ROOT}/{k}' for k in ['data','splits','models','results','logs']}
PATHS['cache']=f'{ROOT}/data/cache'; PATHS['shap']=f'{ROOT}/results/shap'
os.makedirs(PATHS['shap'],exist_ok=True)
CLASSES=['Benign','DDoS','DoS','Mirai','Recon','Spoofing','Web','BruteForce']
class_to_idx={c:i for i,c in enumerate(CLASSES)}; N_CLASSES=len(CLASSES)

# ---- knobs (raise for final) ----
N_PER_CLASS=40      # eval samples per class (×8)
BG_K=40             # k-means background summary points
NSAMPLES=128        # KernelSHAP coalitions per instance (fixed -> deterministic)
print('shap',shap.__version__,'| TF',tf.__version__)

Mounted at /content/drive
shap 0.52.0 | TF 2.20.0


In [2]:
# reload data, split, scaler; build a feature-name list
df=pd.read_parquet(f"{PATHS['cache']}/ciciot_processed.parquet")
FEATURES=[c for c in df.columns if c!='y']
sp=np.load(f"{PATHS['splits']}/split_primary.npz"); tr,te=sp['train'],sp['test']
scaler=joblib.load(f"{PATHS['models']}/scaler_primary.joblib")
Xall=df[FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
y=df['y'].map(class_to_idx).astype('int64').values; del df; gc.collect()
Xtr=scaler.transform(Xall.iloc[tr]).astype('float32'); ytr=y[tr]
Xte=scaler.transform(Xall.iloc[te]).astype('float32'); yte=y[te]
del Xall; gc.collect()
N_FEATURES=Xtr.shape[1]; print('features',N_FEATURES)

features 46


## 1. §4.0 attribution protocol — fixed eval subset, background, unified predict

In [3]:
# Fixed stratified eval subset (seeded) — SAME instances explained for every model
rng=np.random.RandomState(0); idx_sel=[]
for ci in range(N_CLASSES):
    pool=np.where(yte==ci)[0]
    take=min(N_PER_CLASS,len(pool))
    idx_sel.append(rng.choice(pool,size=take,replace=False))
idx_sel=np.concatenate(idx_sel)
Xeval=Xte[idx_sel]; yeval=yte[idx_sel]
print('eval subset',Xeval.shape,'| per-class:',[int((yeval==i).sum()) for i in range(N_CLASSES)])

# Fixed k-means background from TRAIN (seeded)
bg=shap.kmeans(Xtr[rng.choice(len(Xtr),5000,replace=False)], BG_K)

# Unified predict_proba (batched; handles keras + tflite via resize)
def make_predict(path, kind):
    if kind=='keras':
        m=tf.keras.models.load_model(path)
        return lambda Z: m.predict(np.asarray(Z,'float32'),batch_size=4096,verbose=0)
    it=tf.lite.Interpreter(model_path=path)
    inp=it.get_input_details()[0]['index']; outd=it.get_output_details()[0]['index']
    def f(Z):
        Z=np.asarray(Z,'float32'); it.resize_tensor_input(inp,Z.shape); it.allocate_tensors()
        dt=it.get_input_details()[0]['dtype']
        it.set_tensor(inp, Z.astype(dt)); it.invoke()
        P=it.get_tensor(outd).astype('float32'); P=np.clip(P,0,None)
        s=P.sum(1,keepdims=True); s[s==0]=1.0; return P/s
    return f

MODELS={
 'M0_s0': (f"{PATHS['models']}/M0_seed0.keras",'keras'),
 'M0_s1': (f"{PATHS['models']}/M0_seed1.keras",'keras'),
 'M0_s2': (f"{PATHS['models']}/M0_seed2.keras",'keras'),
 'M2_prune50': (f"{PATHS['models']}/M2_prune50.keras",'keras'),
 'M2_prune80': (f"{PATHS['models']}/M2_prune80.keras",'keras'),
 'M3_distilled': (f"{PATHS['models']}/M3_distilled.keras",'keras'),
 'M1_int8': (f"{PATHS['models']}/M1_int8.tflite",'tflite'),
 'M1_float16': (f"{PATHS['models']}/M1_float16.tflite",'tflite'),
}
json.dump({'n_per_class':N_PER_CLASS,'bg_k':BG_K,'nsamples':NSAMPLES,'eval_n':int(len(Xeval)),
           'explainer':'KernelSHAP','attribution_target':'true class','seed':0},
          open(f"{PATHS['logs']}/attribution_protocol.json",'w'),indent=2)
print('protocol logged.')

eval subset (320, 46) | per-class: [40, 40, 40, 40, 40, 40, 40, 40]
protocol logged.


## 2. Compute SHAP (once) — attribute the TRUE class; cache to Drive

In [4]:
def true_class_shap(name, path, kind, explainer_seed=0):
    cache=f"{PATHS['shap']}/{name}_es{explainer_seed}.npy"
    if os.path.exists(cache):
        print('cached',name,'es',explainer_seed); return np.load(cache)
    np.random.seed(explainer_seed)
    f=make_predict(path,kind)
    expl=shap.KernelExplainer(f, bg)
    sv=expl.shap_values(Xeval, nsamples=NSAMPLES, silent=True)
    # robust normalisation -> (n_samples, n_features, n_classes)
    if isinstance(sv, list):                 # older API: list of per-class arrays
        sv=np.stack(sv, axis=-1)
    sv=np.asarray(sv)
    if sv.ndim==2:                           # single-output fallback
        sv=sv[:,:,None]
    # if classes axis is in the middle, move to last
    if sv.shape[-1]!=N_CLASSES and sv.shape[1]==N_CLASSES and sv.shape[2]==N_FEATURES:
        sv=np.transpose(sv,(0,2,1))
    assert sv.shape[0]==len(Xeval) and sv.shape[1]==N_FEATURES, f"unexpected shap shape {sv.shape}"
    nC=sv.shape[2]
    # attribute each sample's TRUE class (clip index if single-output)
    A=np.stack([sv[i,:,min(yeval[i],nC-1)] for i in range(len(yeval))]).astype('float32')
    np.save(cache,A); print('computed',name,'es',explainer_seed,'shap',sv.shape,'->',A.shape); return A

t0=time.time()
A_m0   = true_class_shap('M0_s0', *MODELS['M0_s0'], explainer_seed=0)
A_m0b  = true_class_shap('M0_s0', *MODELS['M0_s0'], explainer_seed=7)   # noise floor
A_s1   = true_class_shap('M0_s1', *MODELS['M0_s1'], explainer_seed=0)
A_s2   = true_class_shap('M0_s2', *MODELS['M0_s2'], explainer_seed=0)
A_var={}
for n in ['M2_prune50','M2_prune80','M3_distilled','M1_int8','M1_float16']:
    A_var[n]=true_class_shap(n, *MODELS[n], explainer_seed=0)
print(f'SHAP done in {(time.time()-t0)/60:.1f} min')

computed M0_s0 es 0 shap (320, 46, 8) -> (320, 46)
computed M0_s0 es 7 shap (320, 46, 8) -> (320, 46)
computed M0_s1 es 0 shap (320, 46, 8) -> (320, 46)
computed M0_s2 es 0 shap (320, 46, 8) -> (320, 46)
computed M2_prune50 es 0 shap (320, 46, 8) -> (320, 46)
computed M2_prune80 es 0 shap (320, 46, 8) -> (320, 46)
computed M3_distilled es 0 shap (320, 46, 8) -> (320, 46)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


computed M1_int8 es 0 shap (320, 46, 8) -> (320, 46)
computed M1_float16 es 0 shap (320, 46, 8) -> (320, 46)
SHAP done in 8.8 min


## 3. §4.1 Agreement metrics vs M0 (+ null band + noise floor)

In [5]:
def agreement(A_ref, A_other):
    # per-sample then average: Spearman, Kendall, Jaccard@k, sign-flip
    sp=[]; kd=[]; j5=[]; j10=[]; sf=[]
    for i in range(len(A_ref)):
        a,b=A_ref[i],A_other[i]
        if np.std(a)<1e-12 or np.std(b)<1e-12:
            continue
        sc=spearmanr(a,b).correlation; kc=kendalltau(a,b).correlation
        if np.isnan(sc) or np.isnan(kc): continue
        sp.append(sc); kd.append(kc)
        for k,store in [(5,j5),(10,j10)]:
            ta=set(np.argsort(-np.abs(a))[:k]); tb=set(np.argsort(-np.abs(b))[:k])
            store.append(len(ta&tb)/len(ta|tb))
        sf.append(np.mean(np.sign(a)!=np.sign(b)))
    return dict(spearman=np.nanmean(sp),kendall=np.nanmean(kd),
               jaccard5=np.mean(j5),jaccard10=np.mean(j10),signflip=np.mean(sf))

# noise floor (M0 vs its own 2nd run) and null band (M0 vs seeds 1,2)
floor=agreement(A_m0,A_m0b)
null1=agreement(A_m0,A_s1); null2=agreement(A_m0,A_s2)
null={k:np.mean([null1[k],null2[k]]) for k in floor}
print('explainer-noise floor (M0 vs M0):', {k:round(v,3) for k,v in floor.items()})
print('seed null band  (M0 vs seeds):  ', {k:round(v,3) for k,v in null.items()})

rows=[{'comparison':'FLOOR_explainer',**floor},{'comparison':'NULL_seed',**null}]
for n,A in A_var.items():
    rows.append({'comparison':n, **agreement(A_m0,A)})
agt=pd.DataFrame(rows); pd.set_option('display.width',200)
print(agt.round(3).to_string(index=False))
agt.to_csv(f"{PATHS['results']}/faithfulness_agreement.csv",index=False)

explainer-noise floor (M0 vs M0): {'spearman': np.float64(0.423), 'kendall': np.float64(0.391), 'jaccard5': np.float64(0.393), 'jaccard10': np.float64(0.338), 'signflip': np.float64(0.222)}
seed null band  (M0 vs seeds):   {'spearman': np.float64(0.558), 'kendall': np.float64(0.522), 'jaccard5': np.float64(0.455), 'jaccard10': np.float64(0.462), 'signflip': np.float64(0.17)}
     comparison  spearman  kendall  jaccard5  jaccard10  signflip
FLOOR_explainer     0.423    0.391     0.393      0.338     0.222
      NULL_seed     0.558    0.522     0.455      0.462     0.170
     M2_prune50     0.642    0.604     0.480      0.521     0.145
     M2_prune80     0.377    0.348     0.324      0.352     0.225
   M3_distilled     0.439    0.407     0.348      0.383     0.206
        M1_int8     0.787    0.758     0.666      0.707     0.086
     M1_float16     0.996    0.995     0.995      0.995     0.001


In [6]:
# Per-class Spearman vs M0 (lower = explanations drifted more for that class)
def perclass_spearman(A_ref,A_other):
    out={}
    for ci,c in enumerate(CLASSES):
        m=np.where(yeval==ci)[0]; vals=[]
        for i in m:
            a,b=A_ref[i],A_other[i]
            if np.std(a)>1e-12 and np.std(b)>1e-12: vals.append(spearmanr(a,b).correlation)
        out[c]=round(float(np.nanmean(vals)),3) if vals else np.nan
    return out
pc=[{'model':n, **perclass_spearman(A_m0,A)} for n,A in A_var.items()]
pcdf=pd.DataFrame(pc); print(pcdf.to_string(index=False))
pcdf.to_csv(f"{PATHS['results']}/faithfulness_perclass_spearman.csv",index=False)

       model  Benign  DDoS   DoS  Mirai  Recon  Spoofing   Web  BruteForce
  M2_prune50   0.641 0.668 0.740  0.709  0.637     0.624 0.628       0.488
  M2_prune80   0.372 0.460 0.537  0.449  0.349     0.371 0.262       0.218
M3_distilled   0.497 0.454 0.489  0.477  0.451     0.418 0.358       0.371
     M1_int8   0.713 0.774 0.610  0.919  0.813     0.836 0.884       0.744
  M1_float16   1.000 0.998 0.994  1.000  0.998     0.997 0.997       0.980


## 4. §4.2 Perturbation faithfulness — AOPC / comprehensiveness (3 maskers)
Delete top-k SHAP features, measure predicted-class probability drop, vs averaged random-k. Three maskers (mean/zero/marginal) test masking-sensitivity.

In [7]:
KS=[1,3,5,10]; RAND_DRAWS=20
mask_values={'mean':Xtr.mean(0), 'zero':np.zeros(N_FEATURES,'float32')}

def mask_fill(masker, feats, seed):
    # mean/zero: fixed vector; marginal: a RANDOM training row (varies with seed)
    if masker in mask_values:
        return mask_values[masker][feats]
    row=Xtr[np.random.RandomState(seed).randint(len(Xtr))]
    return row[feats]

def comprehensiveness(predict, X, A, masker):
    p0=predict(X); pred=p0.argmax(1); base=p0[np.arange(len(X)),pred]
    out={}
    for k in KS:
        Xm=X.copy()
        for i in range(len(X)):
            topk=np.argsort(-np.abs(A[i]))[:k]
            Xm[i,topk]=mask_fill(masker, topk, seed=1000+i)   # marginal varies per sample
        pk=predict(Xm)[np.arange(len(X)),pred]
        out[k]=float(np.mean(base-pk))
    aopc=np.mean(list(out.values()))
    randk={}
    for k in KS:
        drops=[]
        for d in range(RAND_DRAWS):
            r=np.random.RandomState(d); Xm=X.copy()
            for i in range(len(X)):
                rk=r.choice(N_FEATURES,k,replace=False)
                Xm[i,rk]=mask_fill(masker, rk, seed=d*9999+i)
            drops.append(np.mean(base-predict(Xm)[np.arange(len(X)),pred]))
        randk[k]=float(np.mean(drops))
    aopc_rand=np.mean(list(randk.values()))
    return aopc, aopc_rand

rows=[]
for name,(path,kind) in [('M0_s0',MODELS['M0_s0']),('M2_prune50',MODELS['M2_prune50']),
        ('M2_prune80',MODELS['M2_prune80']),('M3_distilled',MODELS['M3_distilled']),
        ('M1_int8',MODELS['M1_int8']),('M1_float16',MODELS['M1_float16'])]:
    A = A_m0 if name=='M0_s0' else A_var[name]
    pred=make_predict(path,kind)
    for msk in ['mean','zero','marginal']:
        a,ar=comprehensiveness(pred,Xeval.copy(),A,msk)
        rows.append({'model':name,'masker':msk,'AOPC_topk':round(a,4),
                     'AOPC_randk':round(ar,4),'gap':round(a-ar,4)})
pf=pd.DataFrame(rows); print(pf.to_string(index=False))
pf.to_csv(f"{PATHS['results']}/faithfulness_perturbation.csv",index=False)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


       model   masker  AOPC_topk  AOPC_randk    gap
       M0_s0     mean     0.5662      0.1667 0.3995
       M0_s0     zero     0.5662      0.1667 0.3995
       M0_s0 marginal     0.5560      0.1599 0.3961
  M2_prune50     mean     0.5903      0.1478 0.4424
  M2_prune50     zero     0.5903      0.1478 0.4424
  M2_prune50 marginal     0.5814      0.1485 0.4329
  M2_prune80     mean     0.5483      0.1014 0.4468
  M2_prune80     zero     0.5483      0.1014 0.4468
  M2_prune80 marginal     0.5098      0.1037 0.4061
M3_distilled     mean     0.6610      0.1833 0.4778
M3_distilled     zero     0.6610      0.1833 0.4778
M3_distilled marginal     0.6287      0.1892 0.4395
     M1_int8     mean     0.5115      0.1449 0.3665
     M1_int8     zero     0.5115      0.1449 0.3665
     M1_int8 marginal     0.4965      0.1403 0.3562
  M1_float16     mean     0.5668      0.1665 0.4003
  M1_float16     zero     0.5668      0.1665 0.4003
  M1_float16 marginal     0.5561      0.1597 0.3964


## 5. Manual commit

In [8]:
ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
print('results:', [f for f in os.listdir(f'{ROOT}/results') if f.startswith('faithfulness')])
# !git -C "{ROOT}" add notebooks/ results/ logs/
# !git -C "{ROOT}" commit -q -m "Phase C: attribution protocol, SHAP agreement + perturbation faithfulness, noise floors"
# !git -C "{ROOT}" push origin main

results: ['faithfulness_agreement.csv', 'faithfulness_perclass_spearman.csv', 'faithfulness_perturbation.csv']


## Read the output
- **Agreement table:** for each variant, Spearman/Jaccard vs M0. A drop **counts** only if it falls below the NULL_seed band (and clears the FLOOR_explainer noise). M2_prune80 should show the largest explanation drift.
- **Per-class Spearman:** which classes' explanations drift most — pair with the calibration collapse (Web/BruteForce/Recon). If the SAME classes degrade in BOTH calibration and faithfulness, that's the unified killer finding.
- **Perturbation table:** `gap = AOPC_topk − AOPC_randk` > 0 means SHAP top-k genuinely drives predictions (faithful). Watch whether the gap **shrinks under compression** (faithfulness loss) and whether the conclusion holds across all 3 maskers (masking-robustness).

**Next: Phase D** — TON_IoT cross-dataset validation (does the pattern transfer?).